## Dataset used: **Avenue Dataset for Abnormal Event Detection**



This dataset accompanies paper "Abnormal Event Detection at 150 FPS in Matlab, Cewu Lu, Jianping Shi, Jiaya Jia, International Conference on Computer Vision, (ICCV), 2013"









# Tracking + Anomaly Detection in Avenue Dataset

#Loading the Avenue Dataset

`NOTE:` This is just a reference code. It is not mandatory to use the same code, make changes as u need

In [ ]:
!wget -O Avenue_Dataset.zip "https://www.cse.cuhk.edu.hk/leojia/projects/detectabnormal/Avenue_Dataset.zip"
!unzip -q Avenue_Dataset.zip -d /content/drive/MyDrive/Avenue_Dataset

In [ ]:
!ls /content/drive/MyDrive/Avenue_Dataset/

output_videos


### Importing required libraries

In [2]:
import os
import cv2
import torch
import shutil
import random
import numpy as np
from glob import glob
from tqdm import tqdm
import matplotlib.pyplot as plt

np.random.seed(42)

### Mount google drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


```
Avenue Dataset
 └── testing_videos
 └── testing_vol
 └── training_videos
 └── training_vol
     
```

In [3]:
if not os.path.exists('yolov5'):
    !git clone https://github.com/ultralytics/yolov5.git

Cloning into 'yolov5'...
remote: Enumerating objects: 18438, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 18438 (delta 86), reused 41 (delta 40), pack-reused 18311 (from 3)
Receiving objects: 100% (18438/18438), 17.52 MiB | 17.82 MiB/s, done.
Resolving deltas: 100% (12531/12531), done.


In [4]:
%cd yolov5/
!pwd

/content/yolov5
/content/yolov5


In [5]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 15.2 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [6]:
%cd /content

/content


#### Use the best YOLOv5 model u have trained

In [7]:
dataset_root = "/content/drive/MyDrive/AvenueDataset"

In [8]:
model_path = "yolov5s.pt"

In [9]:
# Load YOLOv5 model
model = torch.hub.load('yolov5', 'custom', path= model_path, source='local')
model.conf = 0.25
model.iou = 0.45

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


YOLOv5 🚀 v7.0-519-g6be609f3 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

100%|██████████| 14.1M/14.1M [00:00<00:00, 127MB/s] 

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


In [10]:
! pip install deep_sort_realtime opencv-python tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 81.8 MB/s eta 0:00:00


In [11]:
from deep_sort_realtime.deepsort_tracker import DeepSort

In [13]:
output_video_folder = "/content/drive/MyDrive/Avenue_Dataset/output_videos"
os.makedirs(output_video_folder, exist_ok = True)

In [20]:
input_video_path = f"{dataset_root}/testing_videos/09.avi"
output_video_path = os.path.join(output_video_folder, "09_annotated.mp4")

### Tune the below parameters

In [21]:
N_history = 10             # how many frames to use for velocity calc
smoothing_frames = 15      # Smoothing: once RED, stay RED for N frames
velocity_threshold = 300   # Velocity threshold (pixels per second)

In [22]:
# Init DeepSORT tracker
tracker = DeepSort(max_age=30)

# Input video
cap = cv2.VideoCapture(input_video_path)

# Get video info
fps = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video: {fps} FPS, {width}x{height}, {frame_count} frames")

# Prepare output video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

read_count = 0
frame_idx = 0
pbar = tqdm(total=frame_count)

track_memory = {}     # Memory for tracking: store last N positions
anomaly_memory = {}   # store last anomaly state (for smoothing)

saved_anomaly_ids = set()   # to avoid saving duplicates
max_anomaly_frames = 500    # limit on the frames saved

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # YOLO Inference
    results = model(frame)
    detections = results.xyxy[0].cpu().numpy()  # (xmin, ymin, xmax, ymax, conf, cls)

    # Format detections for DeepSORT
    formatted_detections = []
    for d in detections:
        xmin, ymin, xmax, ymax, conf, cls = d
        width_box = xmax - xmin
        height_box = ymax - ymin
        box = [xmin, ymin, width_box, height_box]
        formatted_detections.append([box, conf])

    # Update tracker
    tracks = tracker.update_tracks(formatted_detections, frame=frame)

    # Draw tracks
    for track in tracks:
        if not track.is_confirmed():
            continue

        track_id = track.track_id   # unique ID assigned to the pedestrian
        ltrb = track.to_ltrb()      # gives [left, top, right, bottom] of bounding box
        xmin, ymin, xmax, ymax = map(int, ltrb)

        # Center point
        center_x = (xmin + xmax) / 2
        center_y = (ymin + ymax) / 2

        # Update history
        track_memory.setdefault(track_id, []).append((frame_idx, center_x, center_y))
        track_memory[track_id] = track_memory[track_id][-N_history:]

        # Calculate velocity
        if len(track_memory[track_id]) >= 2:
            f1, x1, y1 = track_memory[track_id][0]
            f2, x2, y2 = track_memory[track_id][-1]

            dt = (f2 - f1) / fps  # time in seconds
            dx = x2 - x1
            dy = y2 - y1
            distance = np.sqrt(dx**2 + dy**2)

            velocity = distance / (dt + 1e-6)  # avoid zero div

            # Anomaly flag
            if velocity > velocity_threshold:
                anomaly = True
            else:
                anomaly = False
        else:
            anomaly = False


        if anomaly:
            anomaly_memory[track_id] = frame_idx  # mark when anomaly seen

        if track_id in anomaly_memory:
            if frame_idx - anomaly_memory[track_id] <= smoothing_frames:
                anomaly = True  # force RED

        # Drawing boxes only around the anomaly
        if anomaly:

          label = f"ID {track_id} {'ANOMALY'}"
          color = (0, 0, 255)

          cv2.rectangle(frame, (xmin, ymin), (xmax, ymax), color, 1)
          cv2.putText(frame, label, (xmin, ymin - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)


    # Write frame
    out.write(frame)

    frame_idx += 1
    read_count += 1
    pbar.update(1)

pbar.close()
cap.release()
out.release()

print(f"Frames read: {read_count} / {frame_count}")
print(f"\n DONE — output saved to: {output_video_path}")

Video: 25.0 FPS, 640x360, 1175 frames


100%|██████████| 1175/1175 [01:35<00:00, 12.32it/s]

Frames read: 1175 / 1175

 DONE — output saved to: /content/drive/MyDrive/Avenue_Dataset/output_videos/09_annotated.mp4




####  Check for different test videos